# Native TDHook workflow

XDRL has no workflow runner of its own. `run_workflow` returns TDHook's native result.

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow, WorkflowResult
from xdrl import BatchSemantics, Interaction, InteractionSpec
from xdrl import KeyRole, KeySchema, ModelRole, TensorDictSchema, run_workflow

policy = TensorDictModule(torch.nn.Linear(4, 2), in_keys=["observation"], out_keys=["action"])
interaction = Interaction(
    policy,
    InteractionSpec(
        ModelRole.ACTOR,
        TensorDictSchema((KeySchema("observation", KeyRole.OBSERVATION),)),
        TensorDictSchema((KeySchema("action", KeyRole.ACTION),)),
        BatchSemantics(("env",)),
    ),
)
data = TensorDict({"observation": torch.randn(8, 4)}, batch_size=[8])
result = run_workflow(interaction, Workflow(ActivationCaching("module")), data)
assert isinstance(result, WorkflowResult)
assert result.plan.model_passes == 1